<div style="text-align:center; border-radius:20px; padding:25px; color:white; margin:0; font-family:sans-serif; background:#1b002a; box-shadow:0px 6px 18px rgba(0,0,0,0.35); margin-bottom:1em;">
          <div style="font-size:200%; color:#FEE100; font-weight:700;">
            amazon-bestsellers-price-vs-user-rating-analysis-ETL,EDA,Visualize
          </div>
        </div>
        

![Dataset Image](https://holosen.net/api/file/fcb7075c-cf32-4726-b6dd-a2617255369e)


In [5]:
import os
main_csv_local_path = 'bestsellers with categories - Data Science Project main.csv'
DATA_DIR = ''
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == main_csv_local_path:
            DATA_DIR = os.path.join(dirname, filename)
            print(DATA_DIR)


In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re

# =============================================================================
# A) Imports + Global Config
# =============================================================================
warnings.filterwarnings('ignore')
np.random.seed(42)

# Config for Kaggle-safe display and plotting
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['axes.titlesize'] = 14

# User inputs
DATASET_NAME = "obaidhere/amazon-bestsellers-price-vs-user-rating-analysis"
EXACT_COLUMNS = ["Name", "Author", "User Rating", "Reviews", "Price", "Year", "Genre"]
TARGET_COL = None 

# =============================================================================
# I) Engineering Helpers
# =============================================================================

def safe_read_csv(path):
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"Error loading CSV: {e}")
        return None

def validate_columns(df, expected):
    if not expected:
        return True
    actual = df.columns.tolist()
    missing = [c for c in expected if c not in actual]
    extra = [c for c in actual if c not in expected]
    if missing or extra:
        print("-" * 30)
        print("COLUMN VALIDATION REPORT")
        if missing: print(f"Missing expected columns: {missing}")
        if extra: print(f"Extra columns found: {extra}")
        print("-" * 30)
    return False if missing else True

def detect_column_types(df):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    return numeric_cols, categorical_cols

def attempt_datetime_detection(df, categorical_cols):
    date_cols = []
    for col in categorical_cols:
        try:
            # Check a sample of 100 non-null values
            sample = df[col].dropna().head(100).astype(str)
            if sample.empty: continue
            # Basic heuristic for date-like strings (contains year or separators)
            # We use errors='coerce' to test parsing
            parsed = pd.to_datetime(sample, errors='coerce')
            if parsed.notnull().mean() > 0.8:
                date_cols.append(col)
        except:
            continue
    return date_cols

# =============================================================================
# B) Load + Validate
# =============================================================================

df = safe_read_csv(DATA_DIR)

if df is not None:
    validate_columns(df, EXACT_COLUMNS)
    
    # =============================================================================
    # C) Data Audit
    # =============================================================================
    print(f"Dataset: {DATASET_NAME}")
    print(f"Shape: {df.shape}")
    print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"Duplicates: {df.duplicated().sum()}")
    
    # Null Audit
    null_report = pd.DataFrame({
        'null_count': df.isnull().sum(),
        'null_pct': (df.isnull().sum() / len(df)) * 100,
        'non_null_count': df.notnull().sum(),
        'dtype': df.dtypes
    })
    print("\n--- NULL AUDIT ---")
    print(null_report)

    numeric_cols, categorical_cols = detect_column_types(df)
    
    # Numeric Stability
    print("\n--- NUMERIC STABILITY ---")
    for col in numeric_cols:
        inf_count = np.isinf(df[col]).sum()
        low_var = df[col].var() < 0.01
        print(f"{col}: Inf/NaN count: {inf_count}, Low Variance: {low_var}")

    # Datetime detection
    detected_dates = attempt_datetime_detection(df, categorical_cols)
    print(f"\nPotential datetime columns: {detected_dates}")

    # =============================================================================
    # D) ETL
    # =============================================================================
    df_clean = df.copy()
    
    # Unified Missing Tokens
    missing_tokens = ["", " ", "NA", "N/A", "null", "None", "nan", "NaN"]
    for col in categorical_cols:
        df_clean[col] = df_clean[col].astype(str).str.strip().replace(missing_tokens, np.nan)
    
    # Deduplication
    dupes_count = df_clean.duplicated().sum()
    df_clean = df_clean.drop_duplicates(keep='first').reset_index(drop=True)
    
    # Numeric Conversion for string-based numbers
    for col in categorical_cols:
        if col not in detected_dates:
            # Check if majority is numeric
            try:
                converted = pd.to_numeric(df_clean[col], errors='coerce')
                if converted.notnull().mean() > 0.5:
                    df_clean[col] = converted
                    print(f"Converted '{col}' to numeric.")
            except:
                pass
    
    # Refresh types after conversion
    numeric_cols, categorical_cols = detect_column_types(df_clean)

    # Imputation + Indicator Creation
    for col in numeric_cols:
        if df_clean[col].isnull().any():
            df_clean[f"{col}__was_missing"] = df_clean[col].isnull().astype(int)
            df_clean[col] = df_clean[col].fillna(df_clean[col].median())
            
    for col in categorical_cols:
        if df_clean[col].isnull().any():
            df_clean[f"{col}__was_missing"] = df_clean[col].isnull().astype(int)
            df_clean[col] = df_clean[col].fillna("Missing")

    # Safe Datetime Conversion
    for col in detected_dates:
        try:
            df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')
        except:
            pass

    # Outlier Analysis (IQR)
    print("\n--- OUTLIER RATE (IQR) ---")
    for col in numeric_cols:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((df_clean[col] < (Q1 - 1.5 * IQR)) | (df_clean[col] > (Q3 + 1.5 * IQR))).sum()
        print(f"{col}: {outliers} outliers ({(outliers/len(df_clean)*100):.2f}%)")
        # Optional winsorization column
        lower = df_clean[col].quantile(0.01)
        upper = df_clean[col].quantile(0.99)
        df_clean[f"{col}__winsor"] = df_clean[col].clip(lower, upper)

    # =============================================================================
    # E) EDA
    # =============================================================================
    print("\n--- NUMERIC SUMMARY ---")
    desc = df_clean[numeric_cols].describe().T
    desc['skew'] = df_clean[numeric_cols].skew()
    desc['kurtosis'] = df_clean[numeric_cols].apply(lambda x: x.kurtosis())
    print(desc)

    print("\n--- CATEGORICAL CARDINALITY ---")
    for col in categorical_cols:
        print(f"{col}: {df_clean[col].nunique()} unique, top: {df_clean[col].value_counts().idxmax()}")

    # Correlations
    if len(numeric_cols) >= 2:
        corr_matrix = df_clean[numeric_cols].corr(method='pearson')
        high_corr = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i):
                if abs(corr_matrix.iloc[i, j]) >= 0.85:
                    high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))
        if high_corr:
            print("\nHigh Correlations (>= 0.85):", high_corr)

    # Target-aware analysis
    if TARGET_COL and TARGET_COL in df_clean.columns:
        print(f"\n--- TARGET ANALYSIS: {TARGET_COL} ---")
        if TARGET_COL in numeric_cols:
            print(df_clean[numeric_cols].corr()[TARGET_COL].sort_values(ascending=False))
        else:
            print(df_clean.groupby(TARGET_COL)[numeric_cols].mean())

    # =============================================================================
    # F) Visualization
    # =============================================================================
    
    # 1. Missingness
    if df.isnull().any().any():
        plt.figure()
        df.isnull().mean().sort_values(ascending=False).head(30).plot(kind='bar', color='steelblue')
        plt.title("Missing Data Percentage by Column")
        plt.ylabel("Fraction of Missing Values")
        plt.tight_layout()
        plt.savefig('missingness_chart.png')

    # 2. Histograms
    num_plots = min(len(numeric_cols), 12)
    if num_plots > 0:
        fig, axes = plt.subplots((num_plots+3)//4, 4, figsize=(18, 4*((num_plots+3)//4)))
        axes = axes.flatten()
        for i, col in enumerate(numeric_cols[:12]):
            sns.histplot(df_clean[col], kde=True, ax=axes[i], color='teal')
            axes[i].set_title(f'Dist: {col}')
        for j in range(i+1, len(axes)): axes[j].axis('off')
        plt.tight_layout()
        plt.savefig('numeric_histograms.png')

    # 3. Boxplots
    if num_plots > 0:
        fig, axes = plt.subplots((num_plots+3)//4, 4, figsize=(18, 4*((num_plots+3)//4)))
        axes = axes.flatten()
        for i, col in enumerate(numeric_cols[:12]):
            sns.boxplot(y=df_clean[col], ax=axes[i], color='salmon')
            axes[i].set_title(f'Outliers: {col}')
        for j in range(i+1, len(axes)): axes[j].axis('off')
        plt.tight_layout()
        plt.savefig('numeric_boxplots.png')

    # 4. Countplots
    cat_plots = [c for c in categorical_cols if df_clean[c].nunique() < 50][:6]
    if cat_plots:
        fig, axes = plt.subplots((len(cat_plots)+2)//3, 3, figsize=(18, 5*((len(cat_plots)+2)//3)))
        axes = axes.flatten()
        for i, col in enumerate(cat_plots):
            order = df_clean[col].value_counts().iloc[:10].index
            sns.countplot(data=df_clean, y=col, order=order, ax=axes[i], hue=col, palette='viridis', legend=False)
            axes[i].set_title(f'Top Values: {col}')
        for j in range(i+1, len(axes)): axes[j].axis('off')
        plt.tight_layout()
        plt.savefig('categorical_counts.png')

    # 5. Correlation Heatmap
    if len(numeric_cols) >= 2:
        plt.figure(figsize=(10, 8))
        # Cap features for heatmap
        heat_cols = numeric_cols[:30]
        sns.heatmap(df_clean[heat_cols].corr(), annot=True, fmt=".2f", cmap='coolwarm', center=0)
        plt.title("Pearson Correlation Heatmap")
        plt.tight_layout()
        plt.savefig('correlation_heatmap.png')

    # =============================================================================
    # G) Feature Engineering
    # =============================================================================
    # Datetime features
    dt_cols = df_clean.select_dtypes(include=['datetime64']).columns
    for col in dt_cols:
        df_clean[f'{col}_year'] = df_clean[col].dt.year
        df_clean[f'{col}_month'] = df_clean[col].dt.month
        df_clean[f'{col}_dayofweek'] = df_clean[col].dt.dayofweek

    # Text features
    for col in categorical_cols:
        if df_clean[col].nunique() > 100: # Heuristic for high cardinality / name-like
            df_clean[f'{col}__len'] = df_clean[col].astype(str).apply(len)
            df_clean[f'{col}__words'] = df_clean[col].astype(str).apply(lambda x: len(x.split()))

    # =============================================================================
    # H) Final Artifact Output
    # =============================================================================
    print("\n--- FINAL PIPELINE SUMMARY ---")
    print(f"Original Shape: {df.shape}")
    print(f"Processed Shape: {df_clean.shape}")
    print(f"Duplicates Removed: {dupes_count}")
    print(f"Columns Created: {len(df_clean.columns) - len(df.columns)}")
    
    # Final data quality stats
    total_missing_after = df_clean.isnull().sum().sum()
    print(f"Total Missing Values in df_clean: {total_missing_after}")
    
    print("\n--- FIRST 5 ROWS OF CLEANED DATA ---")
    print(df_clean.head())

else:
    print("Execution halted: Dataset could not be loaded from DATA_DIR.")


Error loading CSV: [Errno 2] No such file or directory: ''
Execution halted: Dataset could not be loaded from DATA_DIR.
